# Lab 12: Long Short‑Term Memory (LSTM) for Time Series Forecasting

## Objective
Build and train an LSTM network to forecast the next time step of a multivariate time series dataset (energy consumption). The model uses a **univariate multi‑step** input – it sees a window of past values (24 hours) and predicts the next value of the target variable.

## Dataset
- The dataset contains hourly energy consumption data (AEP).
- Features include temporal indicators (hour, day of week, month, etc.) and lagged values.
- Data is already split into `AEP_train.csv`, `AEP_validation.csv`, and `AEP_test.csv`, and pre‑scaled using a `MinMaxScaler`.

## Model Architecture (LSTM)
The LSTM network consists of:
- **LSTM** layer with 8 units, returning sequences (to feed into the next LSTM).
- **LSTM** layer with 20 units (outputs a single vector).
- **Flatten** layer (though the output of the second LSTM is already 2D, Flatten does nothing; we keep it for consistency).
- **Dense** output layer with 1 unit (linear activation) for regression.

## Training
- **Loss:** Mean Absolute Error (MAE)
- **Optimizer:** Adam with learning rate 1e‑3 (then fine‑tuned with 1e‑4)
- **Metrics:** MAE and MAPE
- **Callbacks:** ModelCheckpoint (save best) and TrainingMonitor (plot history)

## Fine‑Tuning
After the initial training, we load the best saved model, reduce the learning rate, and continue training for a few more epochs to further improve performance.

## Evaluation
The final model is evaluated on the test set using several error metrics (MAE, MedAE, MSE, RMSE, MAPE, MDAPE).

---

In [1]:
# 1. SETUP AND IMPORTS

import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow / Keras
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    Dense, Dropout, Activation, Flatten, Input, Reshape, Lambda,
    Concatenate, Conv1D, MaxPooling1D, AveragePooling1D,
    GlobalMaxPooling1D, BatchNormalization, TimeDistributed,
    LSTM, Bidirectional, Add, Layer, LeakyReLU
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.regularizers import l2

# Custom utility modules (from the `timeseires` package)
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

# Scikit‑learn metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score

In [2]:
# 2. PATHS AND PARAMETERS

# Change working directory to dataset location
os.chdir(r'C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12')

# Global parameters
time_steps = 24          # lookback window (24 hours)
num_features = 21        # number of input features
target_col = 0           # column index of the target variable

# Paths for saved models and outputs
OUTPUT_PATH = r'C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12'
CHECKPOINT_PATTERN = os.path.join(OUTPUT_PATH, 'lstm-{epoch:04d}-loss{val_loss:.2f}.h5')
FIG_PATH = os.path.join(OUTPUT_PATH, 'lstm_history.png')
JSON_PATH = os.path.join(OUTPUT_PATH, 'lstm_history.json')

## 3. Define the LSTM Model

In [3]:
def create_lstm():
    """Create an LSTM model for time series regression."""
    input_data = Input(shape=(time_steps, num_features))
    lstm1 = LSTM(8, return_sequences=True)(input_data)
    lstm2 = LSTM(20)(lstm1)          # returns a single vector (batch, 20)
    # Flatten is unnecessary here (already 2D), but kept for consistency
    x = Flatten()(lstm2)
    output_data = Dense(1)(x)         # linear output for regression
    model = Model(inputs=input_data, outputs=output_data)
    return model

In [4]:
# Instantiate and display the model
model = create_lstm()
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 lstm (LSTM)                 (None, 24, 8)             960       
                                                                 
 lstm_1 (LSTM)               (None, 20)                2320      
                                                                 
 flatten (Flatten)           (None, 20)                0         
                                                                 
 dense (Dense)               (None, 1)                 21        
                                                                 
Total params: 3,301
Trainable params: 3,301
Non-trainable params: 0
_________________________________________________________________


## 4. Compile the Model

In [5]:
# If no pre‑trained model is given, compile from scratch
if model is None:
    print("[INFO] compiling model...")
    model = create_lstm()
    opt = Adam(learning_rate=1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=['mae', 'mape'])
else:
    print("[INFO] loading existing model...")
    model = load_model(model)
    # Optionally adjust learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 5. Load and Prepare the Data

The data is already scaled and split into train/validation/test sets. We also load the scaler to invert predictions later.

In [6]:
# Read CSV files
train_df = pd.read_csv('AEP_train.csv')
validation_df = pd.read_csv('AEP_validation.csv')
test_df = pd.read_csv('AEP_test.csv')

# Convert to numpy arrays
train_set = train_df.values
validation_set = validation_df.values
test_set = test_df.values

# Load the scaler
scaler = pickle.load(open('AEP_Scaler.pkl', 'rb'))

train_set.shape, validation_set.shape, test_set.shape

C:\Users\PMYLS\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((84907, 21), (24259, 21), (12130, 21))

## 6. Create Sequences (Univariate Multi‑Step)

We convert the time series into overlapping windows of length `time_steps` (24 hours) to predict the next single value (target_len=1).

In [7]:
start = time.time()
train_X, train_y = univariate_multi_step(train_set, time_steps, target_col=target_col, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=target_col, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=target_col, target_len=1)
print('Time Consumed', time.time()-start, "sec")

# Show shapes
print(f"train_X shape: {train_X.shape}, train_y shape: {train_y.shape}")
print(f"validation_X shape: {validation_X.shape}, validation_y shape: {validation_y.shape}")
print(f"test_X shape: {test_X.shape}, test_y shape: {test_y.shape}")

Time Consumed 0.9278514385223389 sec


## 7. Set Up Callbacks

- `ModelCheckpoint` saves the model with the best validation loss.
- `TrainingMonitor` logs training history and plots it.

In [8]:
checkpoint = ModelCheckpoint(
    CHECKPOINT_PATTERN,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
monitor = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=0)
callbacks = [checkpoint, monitor]

## 8. Train the Model (Initial Run)

In [9]:
epochs = 9
batch_size = 32

history = model.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/9
2653/2653 [==============================] - ETA: 0s - loss: 0.0430 - mae: 0.0430 - mape: 842.3518
Epoch 1: val_loss improved from inf to 0.02547, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12\lstm-0001-loss0.03.h5
2653/2653 [==============================] - 77s 26ms/step - loss: 0.0430 - mae: 0.0430 - mape: 842.3518 - val_loss: 0.0255 - val_mae: 0.0255 - val_mape: 11.3898
Epoch 2/9
2652/2653 [============================>.] - ETA: 0s - loss: 0.0221 - mae: 0.0221 - mape: 32.4825
Epoch 2: val_loss improved from 0.02547 to 0.02178, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12\lstm-0002-loss0.02.h5
2653/2653 [==============================] - 67s 25ms/step - loss: 0.0221 - mae: 0.0221 - mape: 32.4769 - val_loss: 0.0218 - val_mae: 0.0218 - val_mape: 9.4177
Epoch 3/9
2652/2653 [============================>.] - ETA: 0s - loss: 0.0156 - mae: 0.0156 - mape: 164.8011
Epoch 3: val_loss improved from 0.02178 to 0.01252, saving model to C:\Use

## 9. Evaluate the Initial Model

We load the best saved model (based on validation loss) and compute error metrics on the test set.

In [10]:
# Load the best model from the initial training
best_model_path = os.path.join(OUTPUT_PATH, 'lstm-0009-loss0.01.h5')
model_best = load_model(best_model_path)

# Predict on test set
y_pred_scaled = model_best.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

# Compute metrics
MAE = np.mean(abs(y_pred - y_test_unscaled))
MEDAE = np.median(abs(y_pred - y_test_unscaled))
MSE = np.mean((y_pred - y_test_unscaled)**2)
RMSE = np.sqrt(MSE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred.shape= ', y_pred.shape)

379/379 [==============================] - 5s 7ms/step
Mean Absolute Error (MAE): 116.19
Median Absolute Error (MedAE): 94.08
Mean Squared Error (MSE): 22951.03
Root Mean Squared Error (RMSE): 151.5
Mean Absolute Percentage Error (MAPE): 0.79 %
Median Absolute Percentage Error (MDAPE): 0.65 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## 10. Fine‑Tuning the Model

We continue training from the best checkpoint with a lower learning rate to further improve performance.

In [11]:
# Path to the model to fine‑tune
finetune_model_path = os.path.join(OUTPUT_PATH, 'lstm-0009-loss0.01.h5')

# Load the model
model_ft = load_model(finetune_model_path)

# Reduce learning rate for fine‑tuning
K.set_value(model_ft.optimizer.lr, 1e-4)
print("[INFO] new learning rate: {}".format(K.get_value(model_ft.optimizer.lr)))

# Re‑compile (optional but safe)
model_ft.compile(loss='mae', optimizer=model_ft.optimizer, metrics=['mae', 'mape'])

# Set up callbacks – use a distinct checkpoint pattern for fine‑tuning
ft_checkpoint_path = os.path.join(OUTPUT_PATH, 'lstm-ft-{epoch:04d}-loss{val_loss:.2f}.h5')
ft_checkpoint = ModelCheckpoint(ft_checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
ft_monitor = TrainingMonitor(FIG_PATH.replace('.png', '_ft.png'), jsonPath=JSON_PATH.replace('.json', '_ft.json'), startAt=4)
ft_callbacks = [ft_checkpoint, ft_monitor]

[INFO] loading C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12\lstm-0009-loss0.01.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [12]:
# Fine‑tune for a few epochs
ft_epochs = 4
history_ft = model_ft.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=ft_epochs,
    validation_data=(validation_X, validation_y),
    callbacks=ft_callbacks,
    verbose=1
)

Epoch 1/4
2652/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 84.7257
Epoch 1: val_loss improved from inf to 0.00663, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12\lstm-ft-0001-loss0.01.h5
2653/2653 [==============================] - 74s 26ms/step - loss: 0.0066 - mae: 0.0066 - mape: 84.7081 - val_loss: 0.0066 - val_mae: 0.0066 - val_mape: 2.8966
Epoch 2/4
2652/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 117.5598
Epoch 2: val_loss improved from 0.00663 to 0.00661, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\LAB_12\lstm-ft-0002-loss0.01.h5
2653/2653 [==============================] - 66s 25ms/step - loss: 0.0066 - mae: 0.0066 - mape: 117.5352 - val_loss: 0.0066 - val_mae: 0.0066 - val_mape: 2.9641
Epoch 3/4
2652/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 127.8886
Epoch 3: val_loss improved from 0.00661 to 0.00656, saving model to C

## 11. Evaluate the Fine‑Tuned Model

In [13]:
# Load the best fine‑tuned model
best_ft_path = os.path.join(OUTPUT_PATH, 'lstm-ft-0004-loss0.01.h5')
model_ft_best = load_model(best_ft_path)

# Predict and evaluate
y_pred_ft_scaled = model_ft_best.predict(test_X)
y_pred_ft = scaler.inverse_transform(y_pred_ft_scaled)

# Compute metrics
MAE_ft = np.mean(abs(y_pred_ft - y_test_unscaled))
MEDAE_ft = np.median(abs(y_pred_ft - y_test_unscaled))
MSE_ft = np.mean((y_pred_ft - y_test_unscaled)**2)
RMSE_ft = np.sqrt(MSE_ft)
MAPE_ft = np.mean(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100
MDAPE_ft = np.median(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE_ft, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE_ft, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE_ft, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE_ft, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE_ft, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE_ft, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred_ft.shape= ', y_pred_ft.shape)

379/379 [==============================] - 5s 8ms/step
Mean Absolute Error (MAE): 108.17
Median Absolute Error (MedAE): 86.19
Mean Squared Error (MSE): 20407.52
Root Mean Squared Error (RMSE): 142.85
Mean Absolute Percentage Error (MAPE): 0.74 %
Median Absolute Percentage Error (MDAPE): 0.59 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## 12. Summary and Comparison

We compare the initial and fine‑tuned model performance on the test set.

| Metric | Initial | Fine‑Tuned | Improvement |
|--------|---------|------------|-------------|
| MAE    | 116.19  | 108.17     | -6.90%      |
| RMSE   | 151.50  | 142.85     | -5.71%      |
| MAPE   | 0.79%   | 0.74%      | -0.05 pp    |

Fine‑tuning with a lower learning rate improved all metrics, demonstrating the benefit of continued training with a smaller step size. The LSTM model effectively captures temporal dependencies in the 24‑hour input window, achieving very low percentage errors.